In [50]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker

In [51]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_H3K27ac/")

Prepare data

In [52]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")

In [53]:
# sc.pp.normalize_total(rna)
# sc.pp.log1p(rna)
# sc.pp.pca(rna)
# sc.pp.neighbors(rna)
# sc.tl.leiden(rna)

In [54]:
# sc.pl.spatial(rna, spot_size=1, color="leiden")

In [55]:
# spatial = atac.obsm["spatial"].copy()
# s0 = spatial[:,0].copy()
# spatial[:,0] = spatial[:, 1].copy()
# spatial[:,1] = s0
# rna.obsm["spatial"] =  spatial.copy()

In [56]:
(rna.obs_names == atac.obs_names).all()

np.True_

In [57]:
atac.obsm["spatial"] = rna.obsm["spatial"].copy()

In [58]:
# rna.write("rna.h5ad")
# atac.write("atac.h5ad")

In [59]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [60]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(atac, output_file="atac.h5")

In [61]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [62]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_H3K27ac/"
# bm.run(methods=["Seurat_WNN", "Single_modal"],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        n_cluster=16,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_brain_H3K27ac/",
#        hvg_num=3000,
#        )

In [63]:
# methods =  ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",    "scMM",       "scMDC",
#                 "Matilda",   "moETM",  "MISO",   "SpatialGlue",  "COSMOS",     "PRESENT",
#                 "SMOPCA",       "CellCharter"
#                ]
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_brain_H3K27ac/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

In [64]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_brain_H3K27ac/spatialglue.csv", index_col=0)

In [65]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [66]:
# sc.pl.umap(rna, color="cluster")
# sc.pl.spatial(rna, color="cluster", spot_size=1)

Plot

In [67]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_brain_H3K27ac/"
methods = ["Seurat_WNN", "MOFA2", "MultiVI", "Multigrate", "scMM", "scMDC", "Matilda", "moETM",
"MISO", "SpatialGlue",  "COSMOS", "PRESENT", "SMOPCA", "CellCharter"]
res = bm.read_result(path=result_folder,
                     methods=methods + ["rna", "atac"],
                     reindex=False)

2026-04-03 21:41:00 - WARNING - '_latent' result for 'Seurat_WNN' not found at: /mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_brain_H3K27ac/seurat_wnn_latent.csv


In [68]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [rna.obsm["spatial"]]
spatial = transform_coord(spatial, vertical=True, horizontal=False, angle=0)

In [69]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["H3K27ac"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"
bg_dict["Annotation"] = "#97a4af"

In [70]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Mouse_brain_H3K27ac"

In [71]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [72]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 6.8),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=7,
#                 ncol=6,
#                 xlabel=["RNA", "H3K27ac", "Seurat_WNN", "Matilda", "MultiVI",  "MOFA2", "scMDC",  "moETM", "Multigrate", "scMM",
#                 "SpatialGlue", "SMOPCA", "COSMOS", "CellCharter",  "PRESENT",  "MISO"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "atac", "Seurat_WNN", "Matilda", "MultiVI",  "MOFA2", "scMDC",  "moETM", "Multigrate", "scMM",
#                 "SpatialGlue", "SMOPCA", "COSMOS", "CellCharter",  "PRESENT",  "MISO"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True
#                 )

In [73]:
# bm.plot_gene_exp(
#     adata=rna,
#     genes=["Cux1", "Cux2", "Rorb", "Fezf2", "Tle4", "Syt1", "Satb2","Neurod6", "Mef2c",  "Tspan2",  "Sox10", "Mbp",
#     "Gpr88", "Pde10a", "Dlx1", #"Ptprz1", "Car2", "Pou3f2",
#       ],
#     embed=spatial,
#     figsize=(10, 7), 
#     frameon=True,
#     inner_gs_row=1,
#     inner_gs_col=1,
#     ncol=5,
#     xlabel=["Cux1", "Cux2", "Rorb", "Fezf2", "Tle4", "Syt1", "Satb2","Neurod6", "Mef2c",  "Tspan2", "Sox10", "Mbp",
#     "Gpr88", "Pde10a", "Dlx1", #"Ptprz1",  "Car2","Pou3f2", 
#     ],
#     # ylabel=["T cell", "B cell", "Macrophage"],
#     only_show_left=True,
#     axis_width = 1.2,
#     axis_color="lightgrey",
#     outer_row_hspace=0.3,
#     outer_col_wspace=0.05,
#     xlabel_pad=0.012,
#     ylabel_pad=0.02,
#     background_color=lambda x: "#97a4af",
#     sizes=[7.5],
#     vmax="p99",
#     # vmin="p0.5",
#     color_map = "RdYlBu_r",
#     save=f"{figure_save_dir}/marker_gene_plot.pdf"
#     )

In [ ]:
# sc.pp.normalize_total(rna)
# sc.pp.log1p(rna)
# # # sc.pp.scale(rna)

In [75]:
new_adata = sc.AnnData(X=rna[:,["Cux1", "Cux2", "Rorb", "Fezf2", "Tle4", "Syt1", "Satb2","Neurod6", "Mef2c",  "Tspan2", "Sox10", "Mbp",
    "Gpr88", "Pde10a", "Dlx1"]+ list(rna.var_names[3:100])].X.toarray())
new_adata.var_names = rna[:,["Cux1", "Cux2", "Rorb", "Fezf2", "Tle4", "Syt1", "Satb2","Neurod6", "Mef2c",  "Tspan2", "Sox10", "Mbp",
    "Gpr88", "Pde10a", "Dlx1"]+ list(rna.var_names[3:100])].var_names.copy()

In [90]:
# marker_score = bm.cal_marker_score(new_adata, label_dict=res["Cluster"], 
#                                    marker_genes=["Cux1", "Cux2", "Rorb", "Fezf2", "Tle4", "Syt1", "Satb2","Neurod6", "Mef2c",  "Tspan2", "Sox10", "Mbp",
#     "Gpr88", "Pde10a", "Dlx1"],
#                                    re_run=True,
#                                    batch_key="batch")

In [77]:
marker_score = marker_score.drop(["rna", "atac"])

In [78]:
cmap = {
    "Seurat_WNN": "#536ea5", 
    "sciPENN": "#459f6e",
    "PRESENT": "#c5858d",
    "MOFA2": "#f59224",
    "SMOPCA": "#ef85b9",
    "moETM": "#c498c6",
    "Matilda": "#ae6f60",
    "scMDC": "#c9db34",
    "TotalVI": "#a7d38a",
    "SpatialGlue": "#1f4276",
    "CellCharter": "#d1c82e",
    "MultiVI": "#ecae4c",
    "MISO": "#f37597",
    "COSMOS": "#44bbc5",
    "spaMultiVAE": "#931b45",
    "scMM": "#e43a42",
    "Multigrate": "#8dab96"
}

In [89]:
# bm.plot_marker_score(metric_df=marker_score, xlabel=None,
#                      save=f"{figure_save_dir}/marker_score.pdf", vert=True,  label_rotation=30, figsize=(7, 3),
#                      cmap=cmap, ylabel="AUROC")

In [80]:
from benchmarker import cal_chaos_pas

In [81]:
metric = cal_chaos_pas(adata=rna, cluster_dict=res["Cluster"], batch_key=None, return_mean=True)[0]
metric = metric.drop(["rna", "atac"])

In [82]:
# y = [i[0] for i in metric["CHAOS"][:-1]]
y = [i for i in metric["CHAOS"][:-1]]

In [83]:
metric = metric.iloc[:-1,:]

In [84]:
sort_val = marker_score.median(axis=1).sort_values(ascending=False)
sorted_cols = sort_val.index.tolist()

In [85]:
metric = metric.reindex(sorted_cols)

In [88]:
# import matplotlib.pyplot as plt
# import numpy as np

# # 1. 准备数据 (示例数据，请替换为你真实的指标数据)
# methods = sorted_cols

# # 指标 1 (对应第一张图，范围 ~0.6-1.0)
# metric1 = list(metric['CHAOS'])
# # 指标 2 (对应第二张图，范围 ~0.925-0.940)
# metric2 = list(metric['PAS'])

# x = np.arange(len(methods))

# # 2. 创建图表和左侧 Y 轴
# fig, ax1 = plt.subplots(figsize=(7.1, 3), dpi=300)

# # 绘制指标 1 (折线图 1，使用实线和圆形标记)
# color1 = '#7d96bf' # 经典蓝色
# ax1.plot(x, metric1, color=color1, marker='o', linestyle='-', linewidth=1, markersize=3, label='CHAOS')
# ax1.set_ylabel('CHAOS', color=color1, fontsize=12, fontweight='bold', labelpad=15)
# ax1.set_ylim(0.925, 0.94)

# ax1.tick_params(axis='y', labelcolor=color1)

# # 设置 X 轴标签
# ax1.set_xticks(x)
# ax1.set_xticklabels(methods, rotation=30, ha='right', fontsize=11)
# ax1.set_yticks([0.8, 0.9, 1.0])


# # 3. 创建右侧 Y 轴
# ax2 = ax1.twinx()

# # 绘制指标 2 (折线图 2，使用虚线和方形标记以增加区分度)
# color2 = '#c2575c' # 经典红色
# ax2.plot(x, metric2, color=color2, marker='s', linestyle='--', linewidth=1, markersize=3, label='PAS')
# ax2.set_ylabel('PAS', color=color2, fontsize=12, fontweight='bold', labelpad=15)
# ax2.set_ylim(0, 1.0)

# ax2.set_yticks([0.,0.2,0.4, 0.6, 0.8, 1.0])

# ax2.tick_params(axis='y', labelcolor=color2)

# # 4. 合并图例
# lines_1, labels_1 = ax1.get_legend_handles_labels()
# lines_2, labels_2 = ax2.get_legend_handles_labels()

# # 将图例放在图内左上角 (upper left)
# ax1.legend(lines_1 + lines_2, labels_1 + labels_2, 
#            loc='upper right',   # 位置设为左上角
#            frameon=True,       # 开启图例背景框
#            framealpha=0.85,    # 设置背景为 85% 不透明度（半透明），略微透出底部的线
#            edgecolor='none',   # 去除图例边框线，视觉上更干净整洁
#            facecolor='#f8f9fa',# 设置一个极其淡的灰白色背景，增加层次感
#            fontsize=10)

# # 5. 添加网格线以辅助对齐 (可选，建议只开一个轴的网格)
# ax1.grid(True, axis='x', linestyle=':', alpha=0.6)

# # 6. 调整布局并展示
# plt.tight_layout()
# plt.savefig(f"{figure_save_dir}/chaos_pas_methods.pdf") # 保存图片
# plt.show()